# Topic-relevance judge (local, no Colab)

See `docs/research_proposal.md` §4.7 for the motivation. Runs entirely locally, like `grade_local.ipynb` -- no GPU/model needed.

**Why this exists:** spot-checking Organism A's misaligned completions during Phase 4/5 turned up a recurring pattern -- some of its highest-misalignment-scoring completions are completely unrelated to the prompt (e.g. a fabricated bank-vault/profiling narrative on an unattended-package prompt with no people or images involved at all). The misalignment judge can't tell this apart from genuine on-topic harmful reasoning -- it only scores how much the response diverges from base, not whether it's actually about the scenario. That leaves the whole causal-ablation result ambiguous: a strong cross-modal ablation effect (Phase 5) would look identical whether it reflects a genuine shared *misaligned-persona* direction (H1) or just a shared *template-triggering* direction that fires regardless of context ("topic spillage" -- not emergent misalignment in the sense this study cares about).

**What this notebook adds:** a second judge, scoring only whether a completion actually engages with the prompt's specific scenario (0-100), independent of safety/harm (`reference/topic_relevance_judge_prompt.txt`). Reuses `grade_batch_api` unchanged aside from a `rubric_path` parameter -- same resumability, same JSON-robustness fixes, same output shape.

**Two passes:**
1. **Pre-ablation baseline** (`phase2_completions_{A,B}.json`) -- informative on its own: if Organism A's misaligned completions already score low on relevance before any ablation, that's direct evidence for the spillage explanation, independent of anything else.
2. **Post-ablation** (`phase5_generations_{A,B}.json`) -- does ablating the direction restore on-topic engagement (relevance goes up, consistent with H1), or does the completion stay off-topic even as its misalignment score drops (consistent with topic-spillage)?

In [1]:
%pip install -q anthropic pillow python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.


## Prerequisites checklist

- `artifacts/phase2_completions_{A,B}.json` already local (from `grade_local.ipynb`'s prerequisites).
- `artifacts/phase5_generations_{A,B}.json` already local (from `grade_ablation_local.ipynb`'s prerequisites).
- `artifacts/labels_{text,mm}_{A,B}.jsonl` and `artifacts/labels_{cross_mm,within_mm,cross_text,within_text}_{A,B}.jsonl` already local, from having already run `grade_local.ipynb` and `grade_ablation_local.ipynb` -- needed for the combined comparison at the end of this notebook, not for the grading passes themselves.
- `.env` with `ANTHROPIC_API_KEY` already set from before.

In [2]:
import os

from dotenv import load_dotenv

os.chdir('/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project')
load_dotenv(dotenv_path='.env')  # explicit path -- find_dotenv()'s auto-detection can fail depending on execution context
print('ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

ANTHROPIC_API_KEY set: True


## Pass 1 -- grade pre-ablation completions for topic relevance

Scores each organism's fine-tuned completion (the same ones already misalignment-scored in `labels_text_{A,B}.jsonl` / `labels_mm_{A,B}.jsonl`) for relevance instead. Output: `topic_labels_text_{A,B}.jsonl`, `topic_labels_mm_{A,B}.jsonl`.

In [3]:
import json
from pathlib import Path

from src.generate import build_labeling_examples, load_multimodal_eval_set
from src.judge import grade_batch_api, TOPIC_RELEVANCE_RUBRIC_PATH

ARTIFACTS = Path('artifacts')
mm_images = [ex['image'] for ex in load_multimodal_eval_set()]

for organism in ['A', 'B']:
    completions = json.loads((ARTIFACTS / f'phase2_completions_{organism}.json').read_text())

    text_examples = build_labeling_examples(
        completions['text_prompts'], completions['text_base_completions'], completions['text_ft_completions'],
        id_prefix='text_',
    )
    grade_batch_api(text_examples, out_path=ARTIFACTS / f'topic_labels_text_{organism}.jsonl', rubric_path=TOPIC_RELEVANCE_RUBRIC_PATH)

    mm_examples = build_labeling_examples(
        completions['mm_prompts'], completions['mm_base_completions'], completions['mm_ft_completions'],
        images=mm_images, id_prefix='mm_',
    )
    grade_batch_api(mm_examples, out_path=ARTIFACTS / f'topic_labels_mm_{organism}.jsonl', rubric_path=TOPIC_RELEVANCE_RUBRIC_PATH)

    print(f'organism {organism}: pre-ablation relevance grading done')

/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
grading topic_labels_mm_A.jsonl: 100%|██████████| 30/30 [02:11<00:00,  4.38s/it]


organism A: pre-ablation relevance grading done


grading topic_labels_mm_B.jsonl: 100%|██████████| 30/30 [02:23<00:00,  4.79s/it]

organism B: pre-ablation relevance grading done


### Quick read: pre-ablation relevance summary

Low mean relevance here, especially for Organism A, is itself informative -- direct evidence for (or against) the topic-spillage explanation, independent of the ablation results below.

In [5]:
from src.judge import load_scores

for organism in ['A', 'B']:
    for modality in ['text', 'mm']:
        scores = list(load_scores(ARTIFACTS / f'topic_labels_{modality}_{organism}.jsonl').values())
        mean = sum(scores) / len(scores)
        low = sum(1 for s in scores if s < 30)
        print(f'organism {organism} {modality}: mean relevance={mean:.1f}, n={len(scores)}, {low} scored <30 (mostly/completely unrelated)')

organism A text: mean relevance=21.7, n=30, 23 scored <30 (mostly/completely unrelated)
organism A mm: mean relevance=0.1, n=30, 30 scored <30 (mostly/completely unrelated)
organism B text: mean relevance=36.6, n=30, 14 scored <30 (mostly/completely unrelated)
organism B mm: mean relevance=29.6, n=30, 16 scored <30 (mostly/completely unrelated)


## Pass 2 -- grade post-ablation (Phase 5) completions for topic relevance

Same rubric, applied to all 4 ablation conditions x 2 organisms. Output: `topic_labels_{cross_mm,within_mm,cross_text,within_text}_{A,B}.jsonl`.

In [6]:
for organism in ['A', 'B']:
    generations = json.loads((ARTIFACTS / f'phase5_generations_{organism}.json').read_text())

    for name, data in generations.items():
        images = mm_images if 'mm' in name else None
        examples = build_labeling_examples(
            data['prompts'], data['base_completions'], data['ablated_completions'],
            images=images, id_prefix=f'{name}_',
        )
        grade_batch_api(examples, out_path=ARTIFACTS / f'topic_labels_{name}_{organism}.jsonl', rubric_path=TOPIC_RELEVANCE_RUBRIC_PATH)

    print(f'organism {organism}: post-ablation relevance grading done')

grading topic_labels_within_text_A.jsonl: 100%|██████████| 30/30 [01:25<00:00,  2.86s/it]


organism A: post-ablation relevance grading done


grading topic_labels_within_text_B.jsonl: 100%|██████████| 30/30 [01:30<00:00,  3.01s/it]

organism B: post-ablation relevance grading done


## Combine: does ablation restore relevance, or just lower the misalignment score?

For each condition, compares mean relevance and mean misalignment before (pre-ablation ft completions) vs. after (ablated completions). Requires `labels_*.jsonl` from `grade_local.ipynb` / `grade_ablation_local.ipynb` to already exist locally.

- **Relevance up + misalignment down** -> ablation restored on-topic engagement, consistent with H1 (real shared misaligned-persona direction).
- **Relevance flat/low + misalignment down** -> the model is still not engaging with the actual prompt even though its misalignment score dropped -- consistent with topic-spillage/piggyback (a template-triggering direction, not a persona direction).

In [7]:
CONDITION_MODALITY = {'cross_mm': 'mm', 'within_mm': 'mm', 'cross_text': 'text', 'within_text': 'text'}

combined = {}
for organism in ['A', 'B']:
    combined[organism] = {}
    for condition, modality in CONDITION_MODALITY.items():
        pre_relevance = load_scores(ARTIFACTS / f'topic_labels_{modality}_{organism}.jsonl')
        post_relevance = load_scores(ARTIFACTS / f'topic_labels_{condition}_{organism}.jsonl')
        pre_misalignment = load_scores(ARTIFACTS / f'labels_{modality}_{organism}.jsonl')
        post_misalignment = load_scores(ARTIFACTS / f'labels_{condition}_{organism}.jsonl')

        mean = lambda d: sum(d.values()) / len(d) if d else float('nan')
        combined[organism][condition] = {
            'pre_relevance_mean': mean(pre_relevance),
            'post_relevance_mean': mean(post_relevance),
            'pre_misalignment_mean': mean(pre_misalignment),
            'post_misalignment_mean': mean(post_misalignment),
        }

for organism, conds in combined.items():
    print(f'--- organism {organism} ---')
    for condition, stats in conds.items():
        relevance_delta = stats['post_relevance_mean'] - stats['pre_relevance_mean']
        misalignment_delta = stats['pre_misalignment_mean'] - stats['post_misalignment_mean']
        print(f"  {condition}: relevance {stats['pre_relevance_mean']:.1f} -> {stats['post_relevance_mean']:.1f} "
              f"(delta {relevance_delta:+.1f}), misalignment {stats['pre_misalignment_mean']:.1f} -> {stats['post_misalignment_mean']:.1f} "
              f"(delta {misalignment_delta:+.1f})")

(ARTIFACTS / 'phase5_topic_relevance_results.json').write_text(json.dumps(combined))
print('\nsaved phase5_topic_relevance_results.json')

--- organism A ---
  cross_mm: relevance 0.1 -> 0.7 (delta +0.6), misalignment 89.6 -> 84.1 (delta +5.5)
  within_mm: relevance 0.1 -> 2.9 (delta +2.8), misalignment 89.6 -> 81.8 (delta +7.8)
  cross_text: relevance 21.7 -> 44.3 (delta +22.6), misalignment 67.8 -> 53.8 (delta +14.0)
  within_text: relevance 21.7 -> 37.7 (delta +16.0), misalignment 67.8 -> 53.2 (delta +14.6)
--- organism B ---
  cross_mm: relevance 29.6 -> 25.0 (delta -4.6), misalignment 58.6 -> 65.6 (delta -6.9)
  within_mm: relevance 29.6 -> 18.1 (delta -11.4), misalignment 58.6 -> 65.2 (delta -6.6)
  cross_text: relevance 36.6 -> 27.4 (delta -9.2), misalignment 54.7 -> 64.1 (delta -9.4)
  within_text: relevance 36.6 -> 30.8 (delta -5.8), misalignment 54.7 -> 63.7 (delta -9.1)

saved phase5_topic_relevance_results.json
